# setup + imports

In [1]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
from starter_kit import load_training_data, load_json, run_episode, generate_predictions, save_predictions_zip
from starter_kit import DeliverySimulator8x8, ACTION_DELTAS

In [2]:
DATA = Path('data')
torch.manual_seed(42)
np.random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

In [3]:
train_data = load_training_data(DATA)
valscen = load_json(DATA / 'validation_scenarios.json')
testscen = load_json(DATA / 'test_scenarios.json')

N = len(train_data['actions'])
G = train_data['grid'].astype(np.float32)
V = train_data['vector'].astype(np.float32)
y = np.asarray(train_data['actions'], dtype=np.int64) 

# data augmentation

In [4]:
M = 7

def transform(r, c, k):
    if k >= 4:
        r, c = c, r                      
    for _ in range(k % 4):
        r, c = c, M - r                  
    return r, c

def action_map(k):
    amap = {4: 4, 5: 5}                  
    for a in range(4):
        dr, dc = ACTION_DELTAS[a]
        r0, c0 = transform(3, 3, k)
        r1, c1 = transform(3 + dr, 3 + dc, k)
        for b in range(4):
            if ACTION_DELTAS[b] == (r1 - r0, c1 - c0):
                amap[a] = b
    return amap

Gs, Vs, ys = [], [], []
for k in range(8):
    amap = action_map(k)
    for t in train_data['trajectories']:
        s = t['scenario']
        s2 = {
            'walls': [list(transform(w[0], w[1], k)) for w in s['walls']],
            'depots': [list(transform(d[0], d[1], k)) for d in s['depots']],
            'agent_pos': list(transform(s['agent_pos'][0], s['agent_pos'][1], k)),
            'package_location': s['package_location'],
            'destination': s['destination'],
        }
        sim = DeliverySimulator8x8()
        obs = sim.reset(s2)
        for j in range(t['sample_start'], t['sample_end']):
            a = amap[int(train_data['actions'][j])]
            Gs.append(obs['grid'])
            Vs.append(obs['vector'])
            ys.append(a)
            sim.step(a)
            obs = sim.observation()

G = np.array(Gs, dtype=np.float32)       
V = np.array(Vs, dtype=np.float32)     
y = np.array(ys, dtype=np.int64)
N = len(y)
print(G.shape, np.bincount(y))

(42616, 6, 8, 8) [9054 9054 9054 9054 3200 3200]


In [5]:
print(np.array_equal(G[:5327], train_data['grid']), np.allclose(V[:5327], train_data['vector']))

True True


# model

In [6]:
class ActionCNN(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(6, c, 3, padding=1), nn.ReLU(),   
            nn.Conv2d(c, c, 3, padding=1), nn.ReLU(),   
            nn.Conv2d(c, c, 3, padding=1), nn.ReLU(),   
        )
        self.head = nn.Sequential(
            nn.Linear(c * 8 * 8 + 13, 128), nn.ReLU(),
            nn.Linear(128, 6),
        )

    def forward(self, g, v):
        h = self.conv(g)
        h = h.flatten(1)
        h = torch.cat([h, v], dim=1) # dim descrie ce axa vrem sa crestem
        return self.head(h)

model = ActionCNN().to(DEVICE)

In [7]:
NA = 6

def action_fn(observation):
    g = torch.from_numpy(observation['grid'].astype(np.float32)).unsqueeze(0).to(DEVICE)    # (1, 6, 8, 8)
    v = torch.from_numpy(observation['vector'].astype(np.float32)).unsqueeze(0).to(DEVICE)  # (1, 13)
    with torch.no_grad():
        scores = model(g, v)[0].cpu().numpy()
    mask = observation['action_mask']
    for a in range(NA):
        if not mask[a]:
            scores[a] = -1e9
    return int(scores.argmax())

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

Gt = torch.from_numpy(G).to(DEVICE)
Vt = torch.from_numpy(V).to(DEVICE)
yt = torch.from_numpy(y).to(DEVICE)


import copy
best_val = -1
best_state = None
epochs = 100
for epoch in range(epochs):
    model.train()
    perm = np.random.permutation(N)
    total_loss = 0.0
    bs = 128
    for s in range(0, N, bs):
        idx = perm[s : s+bs]
        
        Gb = Gt[idx]
        Vb = Vt[idx]
        yb = yt[idx]

        loss = criterion(model(Gb, Vb), yb)
        total_loss += loss.item() * len(idx)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 5 == 4:
        model.eval()
        ok = 0
        for sc in valscen:
            ok += run_episode(sc, action_fn)['success']
        val = ok / len(valscen)
        print(epoch, round(total_loss / N, 4), val)
        if val > best_val:
            best_val = val
            best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)

4 0.273 0.765
9 0.1854 0.795
14 0.0982 0.795
19 0.0427 0.775
24 0.0216 0.74
29 0.0193 0.77
34 0.0183 0.775
39 0.0108 0.77
44 0.0101 0.74
49 0.0042 0.81
54 0.012 0.775
59 0.0096 0.77
64 0.0116 0.8
69 0.0008 0.815
74 0.0 0.8
79 0.0 0.78
84 0.0 0.775
89 0.0 0.775
94 0.0 0.78
99 0.0 0.775


<All keys matched successfully>

# validare + submission

In [9]:
results = []
for sc in valscen:
    results.append(run_episode(sc, action_fn))
success = [r['success'] for r in results]
print(np.mean(success))

0.815


In [10]:
for k in range(len(results)):
    if not results[k]['success']:
        print(k, results[k]['actions'][:40])

3 [2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3]
9 [0, 4, 2, 1, 2, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2]
11 [3, 3, 0, 4, 2, 1, 2, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2]
17 [0, 0, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3]
18 [2, 2, 4, 3, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2]
19 [2, 0, 2, 4, 3, 1, 3, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
20 [3, 0, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2]
36 [0, 0, 0, 0, 0, 0, 4, 3, 3, 3, 1, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3]
41 [2, 2, 

In [11]:
preds = generate_predictions(testscen, action_fn)

In [12]:
save_predictions_zip(preds, Path('predictions.zip'))